In [ ]:
# 1. Sistem Araçları (COLMAP, FFmpeg, Sanal Ekran, ZBar)
!sudo apt-get update -y
!sudo apt-get install -y ffmpeg xvfb libzbar0

In [ ]:
!pip install /kaggle/input/datasets/yusuftiryaki/wheels/*.whl

In [ ]:
!pip install firebase-admin pyzbar opencv-python open3d plyfile

In [ ]:
# Balyoz 2.0: Dosyayı Python ile açıp içindeki tüm Webrtc tetikleyicilerini siliyoruz
file_path = '/usr/local/lib/python3.12/dist-packages/open3d/__init__.py'

with open(file_path, 'r') as file:
    lines = file.readlines()

with open(file_path, 'w') as file:
    for line in lines:
        # Eğer satırda bu bozuk modül çağrılıyorsa, o satırı 'pass' ile değiştir!
        if 'open3d.visualization.webrtc_server' in line:
            file.write('                pass\n')
        else:
            file.write(line)

print("Open3D'nin bozuk Jupyter başlatıcısı tamamen temizlendi!")

In [ ]:
import os
import cv2
import numpy as np
import open3d as o3d
from pyzbar.pyzbar import decode
import firebase_admin
from firebase_admin import credentials, firestore, storage
import subprocess
import time
import glob
import urllib.parse
import json
from plyfile import PlyData
from tqdm.auto import tqdm
from IPython.display import display, HTML

In [ ]:
# ══════════════════════════════════════════════════════════════
# 🎨 GÖRSEL YARDIMCI FONKSİYONLAR (Progress Bar + Özet Kutuları)
# ══════════════════════════════════════════════════════════════

def summary_box(title, items, color="#2d7d46", icon="🌳"):
    """Kompakt HTML özet kutusu gösterir. items = [(anahtar, değer), ...]"""
    rows = ""
    for k, v in items:
        rows += f"<tr><td style='padding:2px 12px;color:#666;white-space:nowrap'>{k}</td>"
        rows += f"<td style='padding:2px 12px;font-weight:600;color:#222'>{v}</td></tr>"
    html = f"""<div style='border-left:4px solid {color};border-radius:4px;padding:8px 12px;margin:6px 0;
    background:#f8f9fa;max-width:520px;font-family:"SF Mono",monospace;font-size:12.5px'>
    <div style='font-weight:700;color:{color};margin-bottom:6px;font-size:13.5px'>{icon} {title}</div>
    <table style='border-collapse:collapse'>{rows}</table></div>"""
    display(HTML(html))

def step_header(step_num, total, title, icon="⚙️"):
    """Pipeline adımı başlığı gösterir."""
    pct = int(step_num / total * 100)
    display(HTML(f"""<div style='font-family:"SF Mono",monospace;font-size:12.5px;padding:4px 0;color:#444'>
    {icon} <b>Adım {step_num}/{total}</b> — {title}
    <div style='background:#e0e0e0;border-radius:3px;height:6px;width:200px;margin-top:3px'>
    <div style='background:#2d7d46;border-radius:3px;height:6px;width:{pct}%'></div></div></div>"""))

def done_msg(msg):
    """Tamamlanma mesajı."""
    display(HTML(f"<div style='font-family:monospace;font-size:12px;color:#2d7d46;padding:1px 0'>✅ {msg}</div>"))

def warn_msg(msg):
    """Uyarı mesajı."""
    display(HTML(f"<div style='font-family:monospace;font-size:12px;color:#c9710a;padding:1px 0'>⚠️ {msg}</div>"))

def err_msg(msg):
    """Hata mesajı."""
    display(HTML(f"<div style='font-family:monospace;font-size:12px;color:#d32f2f;padding:1px 0'>❌ {msg}</div>"))

In [ ]:
# 4. Nerfstudio Kurulumu
!pip install nerfstudio

In [ ]:
def analyze_video_for_hybrid_plate(video_path):
    """Videoyu tarar: QR kod + ArUco Marker okur, piksel merkez konumlarını toplar."""
    cap = cv2.VideoCapture(video_path)
    total_frames = min(int(cap.get(cv2.CAP_PROP_FRAME_COUNT)), 300)

    aruco_dict = cv2.aruco.getPredefinedDictionary(cv2.aruco.DICT_4X4_50)
    parameters = cv2.aruco.DetectorParameters()
    detector = cv2.aruco.ArucoDetector(aruco_dict, parameters)

    tree_id = None
    marker_pixel_width = None
    frame_count = 0
    aruco_observations = []

    pbar = tqdm(total=total_frames, desc="🔍 Video Tarama", unit="kare",
                bar_format='{l_bar}{bar:25}{r_bar}')

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret or frame_count > 300:
            break

        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

        if tree_id is None:
            decoded_objects = decode(gray)
            for obj in decoded_objects:
                tree_id = obj.data.decode('utf-8')
                pbar.set_postfix_str(f"QR: {tree_id[:20]}")
                break

        corners, ids, _ = detector.detectMarkers(gray)
        if ids is not None:
            c = corners[0][0]
            pw = np.linalg.norm(c[0] - c[1])
            center_x = float(np.mean(c[:, 0]))
            center_y = float(np.mean(c[:, 1]))
            aruco_observations.append((frame_count, center_x, center_y, pw))
            if marker_pixel_width is None:
                marker_pixel_width = pw
            pbar.set_postfix_str(f"QR: {'✓' if tree_id else '—'} | ArUco: {len(aruco_observations)}")

        if tree_id is not None and len(aruco_observations) >= 5:
            break

        frame_count += 1
        pbar.update(1)

    pbar.close()
    cap.release()

    if aruco_observations:
        all_pw = [obs[3] for obs in aruco_observations]
        marker_pixel_width = float(np.median(all_pw))

    summary_box("Video Plaka Analizi", [
        ("QR Kod", tree_id or "Bulunamadı"),
        ("ArUco Gözlem", f"{len(aruco_observations)} kare"),
        ("ArUco Piksel (medyan)", f"{marker_pixel_width:.1f} px" if marker_pixel_width else "—"),
        ("Taranan Kare", f"{frame_count}/{total_frames}"),
    ], color="#1976d2", icon="🔍")

    return tree_id, marker_pixel_width, frame_count, aruco_observations

In [ ]:
import numpy as np
from plyfile import PlyData

def convert_ply_to_splat(ply_file_path, splat_file_path, aruco_3d_pos=None, scale_factor=1.0):
    """PLY -> SPLAT dönüşümü: ArUco merkezli, çok katmanlı filtreleme ile."""
    pbar = tqdm(total=6, desc="📦 PLY → SPLAT", bar_format='{l_bar}{bar:25}{r_bar}')

    # 1. PLY oku
    plydata = PlyData.read(ply_file_path)
    vertex = plydata['vertex']
    num_pts = len(vertex['x'])
    property_names = [p.name for p in vertex.properties]
    pbar.update(1)

    # 2. Koordinatlar
    x_raw = np.nan_to_num(np.array(vertex['x']), nan=0.0, posinf=0.0, neginf=0.0)
    y_raw = np.nan_to_num(np.array(vertex['y']), nan=0.0, posinf=0.0, neginf=0.0)
    z_raw = np.nan_to_num(np.array(vertex['z']), nan=0.0, posinf=0.0, neginf=0.0)

    if aruco_3d_pos is not None and np.linalg.norm(aruco_3d_pos) > 1e-6:
        x_raw -= aruco_3d_pos[0]
        y_raw -= aruco_3d_pos[1]
        z_raw -= aruco_3d_pos[2]
    pbar.update(1)

    # 3. Çok katmanlı filtreleme
    opacity_mask = np.ones(num_pts, dtype=bool)
    if 'opacity' in property_names:
        opacity_raw = np.nan_to_num(np.array(vertex['opacity']))
        opacity_sigmoid = 1.0 / (1.0 + np.exp(-opacity_raw))
        opacity_mask = opacity_sigmoid > 0.15

    scale_mask = np.ones(num_pts, dtype=bool)
    if 'scale_0' in property_names:
        s0 = np.nan_to_num(np.array(vertex['scale_0']))
        s1 = np.nan_to_num(np.array(vertex['scale_1']))
        s2 = np.nan_to_num(np.array(vertex['scale_2']))
        scale_mask = (s0 < 2.0) & (s1 < 2.0) & (s2 < 2.0)

    xy_dist = np.sqrt(x_raw**2 + y_raw**2) * scale_factor
    z_scaled = z_raw * scale_factor
    distance_mask = (xy_dist < 6.0) & (z_scaled > -1.5) & (z_scaled < 18.0)

    valid_mask = opacity_mask & scale_mask & distance_mask

    if np.sum(valid_mask) < 5000:
        distance_mask = (xy_dist < 12.0) & (z_scaled > -3.0) & (z_scaled < 25.0)
        valid_mask = opacity_mask & distance_mask

    x, y, z = x_raw[valid_mask], y_raw[valid_mask], z_raw[valid_mask]
    valid_pts = len(x)
    pbar.update(1)

    # 4. Renk
    SH_C0 = 0.28209479177387814
    if 'f_dc_0' in property_names:
        r = np.clip((np.nan_to_num(np.array(vertex['f_dc_0']))[valid_mask] * SH_C0 + 0.5), 0.0, 1.0) * 255.0
        g = np.clip((np.nan_to_num(np.array(vertex['f_dc_1']))[valid_mask] * SH_C0 + 0.5), 0.0, 1.0) * 255.0
        b = np.clip((np.nan_to_num(np.array(vertex['f_dc_2']))[valid_mask] * SH_C0 + 0.5), 0.0, 1.0) * 255.0
    elif 'red' in property_names:
        r = np.clip(np.nan_to_num(np.array(vertex['red']))[valid_mask], 0, 255)
        g = np.clip(np.nan_to_num(np.array(vertex['green']))[valid_mask], 0, 255)
        b = np.clip(np.nan_to_num(np.array(vertex['blue']))[valid_mask], 0, 255)
    else:
        r, g, b = np.zeros(valid_pts), np.zeros(valid_pts), np.zeros(valid_pts)

    if 'opacity' in property_names:
        opacity_raw = np.nan_to_num(np.array(vertex['opacity']))[valid_mask]
        a = (1.0 / (1.0 + np.exp(-opacity_raw))) * 255.0
    else:
        a = np.ones(valid_pts) * 255.0
    pbar.update(1)

    # 5. Scale + Rotation
    if 'scale_0' in property_names:
        s0 = np.clip(np.nan_to_num(np.array(vertex['scale_0']))[valid_mask], -10.0, 5.0)
        s1 = np.clip(np.nan_to_num(np.array(vertex['scale_1']))[valid_mask], -10.0, 5.0)
        s2 = np.clip(np.nan_to_num(np.array(vertex['scale_2']))[valid_mask], -10.0, 5.0)
        sx, sy, sz = np.exp(s0), np.exp(s1), np.exp(s2)
    else:
        sx, sy, sz = np.ones(valid_pts)*0.01, np.ones(valid_pts)*0.01, np.ones(valid_pts)*0.01

    if 'rot_0' in property_names:
        rw = np.nan_to_num(np.array(vertex['rot_0']))[valid_mask]
        rx = np.nan_to_num(np.array(vertex['rot_1']))[valid_mask]
        ry = np.nan_to_num(np.array(vertex['rot_2']))[valid_mask]
        rz = np.nan_to_num(np.array(vertex['rot_3']))[valid_mask]
        norms = np.sqrt(rw**2 + rx**2 + ry**2 + rz**2)
        norms[norms == 0] = 1e-6
        rw, rx, ry, rz = rw/norms, rx/norms, ry/norms, rz/norms
        qw = np.clip((rw * 128) + 128, 0, 255)
        qx = np.clip((rx * 128) + 128, 0, 255)
        qy = np.clip((ry * 128) + 128, 0, 255)
        qz = np.clip((rz * 128) + 128, 0, 255)
    else:
        qw, qx, qy, qz = np.ones(valid_pts)*255, np.zeros(valid_pts), np.zeros(valid_pts), np.zeros(valid_pts)
    pbar.update(1)

    # 6. Binary yazım
    dt = np.dtype([
        ('x', '<f4'), ('y', '<f4'), ('z', '<f4'),
        ('sx', '<f4'), ('sy', '<f4'), ('sz', '<f4'),
        ('r', 'u1'), ('g', 'u1'), ('b', 'u1'), ('a', 'u1'),
        ('qx', 'u1'), ('qy', 'u1'), ('qz', 'u1'), ('qw', 'u1')
    ])
    splat_data = np.zeros(valid_pts, dtype=dt)
    splat_data['x'], splat_data['y'], splat_data['z'] = x, y, z
    splat_data['sx'], splat_data['sy'], splat_data['sz'] = sx, sy, sz
    splat_data['r'], splat_data['g'] = r.astype(np.uint8), g.astype(np.uint8)
    splat_data['b'], splat_data['a'] = b.astype(np.uint8), a.astype(np.uint8)
    splat_data['qx'], splat_data['qy'] = qx.astype(np.uint8), qy.astype(np.uint8)
    splat_data['qz'], splat_data['qw'] = qz.astype(np.uint8), qw.astype(np.uint8)

    with open(splat_file_path, "wb") as f:
        f.write(splat_data.tobytes())
    pbar.update(1)
    pbar.close()

    removed_pct = (num_pts - valid_pts) / num_pts * 100 if num_pts > 0 else 0
    summary_box("SPLAT Dönüşümü", [
        ("Ham Nokta", f"{num_pts:,}"),
        ("Temiz Nokta", f"{valid_pts:,}"),
        ("Silinen", f"{num_pts - valid_pts:,} (%{removed_pct:.1f})"),
        ("Dosya", os.path.basename(splat_file_path)),
    ], color="#e65100", icon="📦")

    return splat_file_path

In [ ]:
def calculate_fractal_dimension(pcd_crown):
    """Taç yapısının 3D Kutu Sayma yöntemiyle fraktal skorunu hesaplar."""
    points = np.asarray(pcd_crown.points)
    if len(points) == 0:
        return 0.0

    points_min = points.min(axis=0)
    points_max = points.max(axis=0)
    span = points_max - points_min
    span[span == 0] = 1e-6
    points_normalized = (points - points_min) / span

    scales = np.logspace(1, 8, num=8, endpoint=True, base=2)
    Ns = []
    for scale in scales:
        grid_indices = np.floor(points_normalized * scale).astype(int)
        unique_boxes = len(np.unique(grid_indices, axis=0))
        Ns.append(unique_boxes)

    coeffs = np.polyfit(np.log(scales), np.log(Ns), 1)
    return coeffs[0]

In [ ]:
from scipy.spatial import ConvexHull
from scipy.spatial.distance import pdist

def calculate_additional_metrics(pcd, height):
    """3D nokta bulutundan ek ağaç metriklerini hesaplar (sessiz mod)."""
    points = np.asarray(pcd.points)
    crown_threshold = max(0.3, height * 0.15)

    # 1. Taç çapı
    crown_mask = points[:, 2] > crown_threshold
    crown_points = points[crown_mask]
    crown_diameter_m = 0.0
    if len(crown_points) > 10:
        crown_xy = crown_points[:, :2]
        xy_min, xy_max = crown_xy.min(axis=0), crown_xy.max(axis=0)
        crown_diameter_m = float(np.max(xy_max - xy_min))
        try:
            hull_2d = ConvexHull(crown_xy)
            hull_vertices = crown_xy[hull_2d.vertices]
            crown_diameter_m = float(np.percentile(pdist(hull_vertices), 95))
        except Exception:
            pass

    # 2. Taç yüzey alanı
    crown_surface_area_m2 = 0.0
    crown_pcd = pcd.select_by_index(np.where(crown_mask)[0])
    try:
        hull_mesh, _ = crown_pcd.compute_convex_hull()
        crown_surface_area_m2 = float(hull_mesh.get_surface_area())
    except Exception:
        pass

    # 3. Gövde eğikliği
    trunk_mask = (points[:, 2] > 0.1) & (points[:, 2] < crown_threshold)
    trunk_points = points[trunk_mask]
    trunk_lean_deg = 0.0
    if len(trunk_points) > 20:
        trunk_centered = trunk_points - np.median(trunk_points, axis=0)
        cov_matrix = np.cov(trunk_centered.T)
        eigenvalues, eigenvectors = np.linalg.eigh(cov_matrix)
        principal_axis = eigenvectors[:, np.argmax(eigenvalues)]
        cos_angle = abs(np.dot(principal_axis, [0, 0, 1])) / np.linalg.norm(principal_axis)
        trunk_lean_deg = float(np.degrees(np.arccos(np.clip(cos_angle, -1.0, 1.0))))

    # 4. Taç asimetrisi
    asymmetry_index = 0.0
    if len(crown_points) > 10 and len(trunk_points) > 10:
        asymmetry_m = float(np.linalg.norm(
            np.median(crown_points[:, :2], axis=0) - np.median(trunk_points[:, :2], axis=0)
        ))
        asymmetry_index = asymmetry_m / crown_diameter_m if crown_diameter_m > 0 else 0.0

    # 5. Taç yoğunluğu
    crown_density = 0.0
    try:
        hull_mesh_vol, _ = crown_pcd.compute_convex_hull()
        crown_vol = hull_mesh_vol.get_volume()
        crown_density = float(len(crown_points) / crown_vol) if crown_vol > 0 else 0.0
    except Exception:
        pass

    # 6. Taç tabanı yüksekliği
    z_bins = np.linspace(0, height, num=50)
    z_counts = np.histogram(points[:, 2], bins=z_bins)[0]
    threshold = len(points) * 0.05 / len(z_bins)
    crown_base_height = crown_threshold
    for i in range(len(z_counts)):
        if z_counts[i] > threshold and z_bins[i] > 0.3:
            crown_base_height = float(z_bins[i])
            break

    return {
        "crown_diameter_m": round(crown_diameter_m, 2),
        "crown_surface_area_m2": round(crown_surface_area_m2, 2),
        "trunk_lean_deg": round(trunk_lean_deg, 1),
        "crown_asymmetry_index": round(asymmetry_index, 3),
        "crown_density_pts_m3": round(crown_density, 0),
        "crown_base_height_m": round(crown_base_height, 2),
    }

In [ ]:
import math

def find_aruco_3d_position(transforms_json_path, aruco_observations):
    """ArUco marker'ın 3D konumunu triangülasyon ile hesaplar."""
    if not aruco_observations or len(aruco_observations) < 2:
        warn_msg("Yeterli ArUco gözlemi yok, merkez (0,0,0) kullanılacak.")
        return np.array([0.0, 0.0, 0.0])

    with open(transforms_json_path, 'r') as f:
        data = json.load(f)

    fl_x = data["fl_x"]
    fl_y = data.get("fl_y", fl_x)
    cx = data.get("cx", data.get("w", 1920) / 2)
    cy = data.get("cy", data.get("h", 1080) / 2)

    frame_transforms = {}
    for frame_info in data["frames"]:
        fp = frame_info["file_path"]
        fname = os.path.splitext(os.path.basename(fp))[0]
        try:
            idx = int(fname.split("_")[-1]) - 1
            frame_transforms[idx] = np.array(frame_info["transform_matrix"])
        except (ValueError, IndexError):
            continue

    ray_origins, ray_directions = [], []

    for (frame_idx, px, py, _) in aruco_observations:
        if frame_idx in frame_transforms:
            T = frame_transforms[frame_idx]
        else:
            closest_idx = min(frame_transforms.keys(), key=lambda k: abs(k - frame_idx))
            if abs(closest_idx - frame_idx) > 10:
                continue
            T = frame_transforms[closest_idx]

        d_cam = np.array([(px - cx) / fl_x, (py - cy) / fl_y, 1.0])
        d_cam = d_cam / np.linalg.norm(d_cam)
        R, t = T[:3, :3], T[:3, 3]
        d_world = R @ d_cam
        d_world = d_world / np.linalg.norm(d_world)
        ray_origins.append(t)
        ray_directions.append(d_world)

    if len(ray_origins) < 2:
        warn_msg("Yeterli ışın oluşturulamadı, merkez (0,0,0) kullanılacak.")
        return np.array([0.0, 0.0, 0.0])

    A = np.zeros((3, 3))
    b = np.zeros(3)
    for o, d in zip(ray_origins, ray_directions):
        P = np.eye(3) - np.outer(d, d)
        A += P
        b += P @ o

    try:
        aruco_3d = np.linalg.solve(A, b)
        done_msg(f"ArUco 3D Konum: ({aruco_3d[0]:.3f}, {aruco_3d[1]:.3f}, {aruco_3d[2]:.3f}) — {len(ray_origins)} ışın")
        return aruco_3d
    except np.linalg.LinAlgError:
        warn_msg("Triangülasyon çözülemedi, merkez (0,0,0) kullanılacak.")
        return np.array([0.0, 0.0, 0.0])


def calculate_scale_factor(transforms_json_path, aruco_observations, pixel_width, real_width_m=0.15):
    """Birden fazla karedeki ArUco gözlemlerinden robust ölçek çarpanı hesaplar."""
    with open(transforms_json_path, 'r') as f:
        data = json.load(f)

    fl_x = data["fl_x"]

    frame_transforms = {}
    for frame_info in data["frames"]:
        fp = frame_info["file_path"]
        fname = os.path.splitext(os.path.basename(fp))[0]
        try:
            idx = int(fname.split("_")[-1]) - 1
            frame_transforms[idx] = np.array(frame_info["transform_matrix"])
        except (ValueError, IndexError):
            continue

    scale_factors = []
    for (frame_idx, px, py, pw) in aruco_observations:
        real_distance_m = (fl_x * real_width_m) / pw
        if frame_idx in frame_transforms:
            T = frame_transforms[frame_idx]
        else:
            closest_idx = min(frame_transforms.keys(), key=lambda k: abs(k - frame_idx))
            if abs(closest_idx - frame_idx) > 10:
                continue
            T = frame_transforms[closest_idx]
        virtual_distance = np.linalg.norm(T[:3, 3])
        if virtual_distance > 1e-6:
            scale_factors.append(real_distance_m / virtual_distance)

    if not scale_factors:
        warn_msg("Hiçbir kare eşleştirilemedi! Varsayılan çarpan 1.0")
        return 1.0

    scale_factor = float(np.median(scale_factors))
    cv = float(np.std(scale_factors)) / float(np.mean(scale_factors)) if np.mean(scale_factors) > 0 else 0

    if cv > 0.3:
        q1, q3 = np.percentile(scale_factors, [25, 75])
        iqr = q3 - q1
        filtered = [sf for sf in scale_factors if q1 - 1.5*iqr <= sf <= q3 + 1.5*iqr]
        if filtered:
            scale_factor = float(np.median(filtered))

    summary_box("Ölçek Çarpanı", [
        ("Scale Factor", f"{scale_factor:.4f}"),
        ("Kare Sayısı", f"{len(scale_factors)}"),
        ("Varyasyon (CV)", f"{cv:.1%}"),
    ], color="#0277bd", icon="📏")

    return scale_factor

In [ ]:
def measure_tree_metrics(ply_file_path, scale_factor, aruco_3d_pos=None):
    """3D nokta bulutundan ağaç metriklerini hesaplar. ArUco merkezli, çok aşamalı gürültü temizleme."""
    pbar = tqdm(total=9, desc="📐 Ağaç Ölçümü", bar_format='{l_bar}{bar:25}{r_bar}')

    # 1. Yükle
    pcd = o3d.io.read_point_cloud(ply_file_path)
    points_initial = len(pcd.points)
    pbar.set_postfix_str(f"{points_initial:,} nokta yüklendi")
    pbar.update(1)

    # 2. ArUco merkezleme
    if aruco_3d_pos is not None and np.linalg.norm(aruco_3d_pos) > 1e-6:
        pcd.translate(-aruco_3d_pos)
    else:
        pcd.translate(-pcd.get_center())
    pbar.update(1)

    # 3. Ölçekleme
    pcd.scale(scale_factor, center=np.array([0.0, 0.0, 0.0]))
    pbar.update(1)

    # 4. Gürültü temizleme (5 alt aşama)
    pbar.set_postfix_str("Gürültü temizleniyor...")
    points_before = len(pcd.points)

    # 4a: Makro filtre
    points = np.asarray(pcd.points)
    xy_dist = np.sqrt(points[:, 0]**2 + points[:, 1]**2)
    macro_mask = (xy_dist < 8.0) & (points[:, 2] > -2.0) & (points[:, 2] < 20.0)
    pcd = pcd.select_by_index(np.where(macro_mask)[0])

    # 4b: İstatistiksel (gevşek)
    cl, ind = pcd.remove_statistical_outlier(nb_neighbors=30, std_ratio=2.0)
    pcd = pcd.select_by_index(ind)

    # 4c: Radius
    cl, ind = pcd.remove_radius_outlier(nb_points=10, radius=0.05)
    pcd = pcd.select_by_index(ind)

    # 4d: DBSCAN
    labels = np.array(pcd.cluster_dbscan(eps=0.08, min_points=15, print_progress=False))
    if len(labels) > 0 and labels.max() >= 0:
        unique_labels, counts = np.unique(labels[labels >= 0], return_counts=True)
        largest_cluster = unique_labels[np.argmax(counts)]
        pcd = pcd.select_by_index(np.where(labels == largest_cluster)[0])

    # 4e: İstatistiksel (sıkı)
    if len(pcd.points) > 100:
        cl, ind = pcd.remove_statistical_outlier(nb_neighbors=50, std_ratio=1.5)
        pcd = pcd.select_by_index(ind)

    pbar.set_postfix_str(f"{points_before:,} → {len(pcd.points):,}")
    pbar.update(1)

    # 5. Zemin tespiti (RANSAC)
    points = np.asarray(pcd.points)
    z_coords = points[:, 2]
    z_threshold = np.percentile(z_coords, 20)
    ground_candidates_idx = np.where(z_coords < z_threshold)[0]

    if len(ground_candidates_idx) > 50:
        ground_pcd = pcd.select_by_index(ground_candidates_idx)
        try:
            plane_model, inliers = ground_pcd.segment_plane(distance_threshold=0.05, ransac_n=3, num_iterations=1000)
            a, b, c, d = plane_model
            ground_z = -d / c if abs(c) > 0.5 else np.percentile(z_coords, 2)
        except Exception:
            ground_z = np.percentile(z_coords, 2)
    else:
        ground_z = np.percentile(z_coords, 2)

    pcd.translate(np.array([0, 0, -ground_z]))
    pbar.update(1)

    # 6. Yükseklik
    points_shifted = np.asarray(pcd.points)
    z_shifted = points_shifted[:, 2]
    above_ground = z_shifted > -0.1
    pcd = pcd.select_by_index(np.where(above_ground)[0])
    points_shifted = np.asarray(pcd.points)
    z_shifted = points_shifted[:, 2]

    tree_height_m = float(np.percentile(z_shifted, 99))

    if tree_height_m > 15.0:
        warn_msg(f"Boy çok yüksek ({tree_height_m:.2f}m)! Ölçek çarpanı kontrol edilmeli.")
    elif tree_height_m < 0.5:
        warn_msg(f"Boy çok düşük ({tree_height_m:.2f}m)! Model bozuk olabilir.")
    pbar.update(1)

    # 7. Taç hacmi
    z_bins = np.linspace(0, tree_height_m, num=40)
    z_hist, _ = np.histogram(z_shifted, bins=z_bins)
    median_density = np.median(z_hist[z_hist > 0]) if np.any(z_hist > 0) else 0
    crown_start_z = 0.5
    for i in range(len(z_hist)):
        if z_hist[i] > median_density * 0.5 and z_bins[i] > 0.3:
            crown_start_z = float(z_bins[i])
            break

    crown_indices = np.where(z_shifted > crown_start_z)[0]
    crown_pcd = pcd.select_by_index(crown_indices)
    crown_volume_m3 = 0.0
    if len(crown_pcd.points) > 50:
        try:
            hull, _ = crown_pcd.compute_convex_hull()
            crown_volume_m3 = hull.get_volume()
        except Exception:
            pass
    pbar.update(1)

    # 8. Gövde çapı (çoklu dilim)
    trunk_diameters = []
    for z_low, z_high in [(0.2, 0.4), (0.3, 0.5), (0.4, 0.6)]:
        if z_high > tree_height_m * 0.5:
            continue
        slice_idx = np.where((z_shifted > z_low) & (z_shifted < z_high))[0]
        slice_pts = np.asarray(pcd.select_by_index(slice_idx).points)[:, :2]
        if len(slice_pts) > 15:
            center_2d = np.median(slice_pts, axis=0)
            distances = np.linalg.norm(slice_pts - center_2d, axis=1)
            q1, q3 = np.percentile(distances, [25, 75])
            valid = distances <= q3 + 1.5 * (q3 - q1)
            if np.sum(valid) > 10:
                trunk_diameters.append((np.median(distances[valid]) * 2) * 100)

    trunk_diameter_cm = float(np.median(trunk_diameters)) if trunk_diameters else 0.0
    pbar.update(1)

    # 9. Fraktal + ek metrikler
    pbar.set_postfix_str("Fraktal & ek metrikler...")
    fractal_score = calculate_fractal_dimension(crown_pcd)
    additional_metrics = calculate_additional_metrics(pcd, tree_height_m)
    pbar.update(1)
    pbar.close()

    # Özet kutu
    summary_box("Ağaç Ölçüm Sonuçları", [
        ("Boy (P99)", f"{tree_height_m:.2f} m"),
        ("Taç Hacmi", f"{crown_volume_m3:.2f} m³"),
        ("Gövde Çapı", f"{trunk_diameter_cm:.1f} cm"),
        ("Fraktal Boyut", f"{fractal_score:.3f}"),
        ("Taç Başlangıcı", f"{crown_start_z:.2f} m"),
        ("Temiz Nokta", f"{len(pcd.points):,} / {points_initial:,}"),
        ("Taç Çapı", f"{additional_metrics['crown_diameter_m']:.2f} m"),
        ("Gövde Eğikliği", f"{additional_metrics['trunk_lean_deg']:.1f}°"),
    ], color="#2d7d46", icon="🌳")

    return tree_height_m, crown_volume_m3, trunk_diameter_cm, fractal_score, ground_z, pcd, additional_metrics

In [ ]:
def analyze_video_color(video_path, n_frames=20):
    """Video karelerinden ağacın renk analizini yapar (HSV segmentasyon)."""
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        err_msg("Video açılamadı!")
        return _empty_color_metrics()

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total_frames <= 0:
        cap.release()
        err_msg("Video kare sayısı okunamadı!")
        return _empty_color_metrics()

    start_idx = int(total_frames * 0.1)
    end_idx = int(total_frames * 0.9)
    frame_indices = np.linspace(start_idx, end_idx, n_frames, dtype=int)

    frame_green_fractions = []
    frame_greenness_indices = []
    frame_stress_ratios = []
    frame_color_homogeneities = []
    frame_trunk_leaf_ratios = []
    all_dominant_colors = []

    for idx in tqdm(frame_indices, desc="🎨 Renk Analizi", unit="kare",
                    bar_format='{l_bar}{bar:25}{r_bar}'):
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if not ret:
            continue

        hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)

        sky_mask = (
            ((hsv[:,:,0] > 90) & (hsv[:,:,0] < 135) & (hsv[:,:,1] < 80)) |
            ((hsv[:,:,1] < 30) & (hsv[:,:,2] > 200))
        )
        ground_mask = (hsv[:,:,2] < 40)
        tree_mask = ~sky_mask & ~ground_mask
        tree_pixels_bgr = frame[tree_mask]

        if len(tree_pixels_bgr) < 100:
            continue

        tree_pixels_rgb = tree_pixels_bgr[:, ::-1].astype(np.float32)
        R, G, B = tree_pixels_rgb[:, 0], tree_pixels_rgb[:, 1], tree_pixels_rgb[:, 2]
        rgb_sum = R + G + B + 1e-6

        greenness = G / rgb_sum
        frame_greenness_indices.append(float(np.mean(greenness)))

        tree_hsv = hsv[tree_mask]
        green_mask = (tree_hsv[:, 0] > 25) & (tree_hsv[:, 0] < 90) & (tree_hsv[:, 1] > 40) & (tree_hsv[:, 2] > 40)
        frame_green_fractions.append(float(np.sum(green_mask)) / len(tree_hsv) if len(tree_hsv) > 0 else 0.0)

        yellow_brown_mask = (
            ((tree_hsv[:, 0] > 15) & (tree_hsv[:, 0] < 35) & (tree_hsv[:, 1] > 40)) |
            ((tree_hsv[:, 0] > 8) & (tree_hsv[:, 0] < 20) & (tree_hsv[:, 1] > 30) & (tree_hsv[:, 2] < 150))
        )
        frame_stress_ratios.append(float(np.sum(yellow_brown_mask)) / len(tree_hsv) if len(tree_hsv) > 0 else 0.0)

        color_std = np.mean([
            np.std(R) / (np.mean(R) + 1e-6),
            np.std(G) / (np.mean(G) + 1e-6),
            np.std(B) / (np.mean(B) + 1e-6)
        ])
        frame_color_homogeneities.append(float(color_std))

        trunk_mask_c = (tree_hsv[:, 1] < 50) & (tree_hsv[:, 2] > 40) & (tree_hsv[:, 2] < 180)
        frame_trunk_leaf_ratios.append(float(np.sum(trunk_mask_c)) / (float(np.sum(green_mask)) + 1e-6))

        all_dominant_colors.append(tree_pixels_rgb.mean(axis=0).tolist())

    cap.release()

    if len(frame_greenness_indices) == 0:
        warn_msg("Hiçbir karede yeterli ağaç pikseli bulunamadı!")
        return _empty_color_metrics()

    green_fraction = float(np.median(frame_green_fractions))
    greenness_index = float(np.median(frame_greenness_indices))
    stress_ratio = float(np.median(frame_stress_ratios))
    color_homogeneity = float(np.median(frame_color_homogeneities))
    trunk_leaf_ratio = float(np.median(frame_trunk_leaf_ratios))
    avg_dominant_color = np.mean(all_dominant_colors, axis=0).tolist() if all_dominant_colors else [0, 0, 0]

    health_score = min(100, max(0, int(
        green_fraction * 60 + (1 - stress_ratio) * 25 + (1 - color_homogeneity) * 15
    )))

    results = {
        "green_fraction": round(green_fraction, 3),
        "greenness_index": round(greenness_index, 4),
        "stress_ratio": round(stress_ratio, 3),
        "color_homogeneity": round(color_homogeneity, 3),
        "trunk_leaf_ratio": round(trunk_leaf_ratio, 2),
        "health_score": health_score,
        "dominant_color_rgb": [round(c, 1) for c in avg_dominant_color],
        "analyzed_frames": len(frame_greenness_indices)
    }

    # Dominant renk kutucuğu
    r_c, g_c, b_c = int(avg_dominant_color[0]), int(avg_dominant_color[1]), int(avg_dominant_color[2])
    color_swatch = f"<span style='display:inline-block;width:12px;height:12px;background:rgb({r_c},{g_c},{b_c});border-radius:2px;vertical-align:middle;margin-right:4px'></span>({r_c}, {g_c}, {b_c})"

    summary_box("Video Renk Analizi", [
        ("Sağlık Skoru", f"{health_score}/100"),
        ("Yeşil Oran", f"{green_fraction:.1%}"),
        ("Stres Oranı", f"{stress_ratio:.1%}"),
        ("Homojenlik", f"{color_homogeneity:.3f}"),
        ("Gövde/Yaprak", f"{trunk_leaf_ratio:.2f}"),
        ("Dominant Renk", color_swatch),
        ("Analiz Edilen", f"{len(frame_greenness_indices)}/{n_frames} kare"),
    ], color="#7b1fa2", icon="🎨")

    return results


def _empty_color_metrics():
    return {
        "green_fraction": 0.0, "greenness_index": 0.0, "stress_ratio": 0.0,
        "color_homogeneity": 0.0, "trunk_leaf_ratio": 0.0, "health_score": 0,
        "dominant_color_rgb": [0, 0, 0], "analyzed_frames": 0
    }

In [ ]:
# --- 1. FIREBASE BAĞLANTISI ---
# Firebase konsolundan indirdiğiniz serviceAccountKey.json dosyasını Colab'a yükleyin
CREDENTIALS_PATH = "/kaggle/input/datasets/yusuftiryaki/firabase-keys/firebase.json"
STORAGE_BUCKET = "studio-166104040-52130.firebasestorage.app" # Kendi bucket isminizle değiştirin

if not firebase_admin._apps:
    cred = credentials.Certificate(CREDENTIALS_PATH)
    firebase_admin.initialize_app(cred, {
        'storageBucket': STORAGE_BUCKET
    })

db = firestore.client()
bucket = storage.bucket()

# Sistem Sabitleri
QR_GERCEK_BOYUT_METRE = 0.15  # 15 cm x 15 cm QR kod kullandığımızı varsayıyoruz
WORKSPACE_DIR = "/kaggle/working/pistachio_workspace"
os.makedirs(WORKSPACE_DIR, exist_ok=True)


In [ ]:
def run_colmap_processing(video_path, doc_id, colmap_dir):
    """FFmpeg + COLMAP ile kamera pozisyonlarını hesaplar."""
    colmap_db_path = os.path.join(colmap_dir, "colmap", "database.db")
    if os.path.exists(colmap_db_path):
        done_msg("COLMAP veritabanı zaten mevcut, atlanıyor.")
        return

    try:
        cmd = [
            "xvfb-run", "-a",
            "ns-process-data", "video",
            "--data", video_path,
            "--output-dir", colmap_dir,
            "--num-frames-target", "60",
            "--matching-method", "sequential",
            "--verbose"
        ]
        env = os.environ.copy()
        env["QT_QPA_PLATFORM"] = "offscreen"
        result = subprocess.run(cmd, check=True, capture_output=True, text=True, env=env)
        done_msg("COLMAP tamamlandı.")
    except subprocess.CalledProcessError as e:
        err_msg(f"COLMAP başarısız! (exit: {e.returncode})")
        # Son 5 satır hata çıktısı göster
        stderr_lines = (e.stderr or "").strip().split('\n')
        for line in stderr_lines[-5:]:
            print(f"  {line}")
        raise Exception(f"ns-process-data komutu başarısız oldu: {e}")

In [ ]:
def run_3d_reconstruction_pipeline(video_path, doc_id):
    """COLMAP + 3DGS + PLY export pipeline'ı."""
    base_dir = f"{WORKSPACE_DIR}/{doc_id}"
    colmap_dir = f"{base_dir}/colmap_data"
    export_dir = f"{base_dir}/export"
    os.makedirs(colmap_dir, exist_ok=True)
    os.makedirs(export_dir, exist_ok=True)

    # Adım 1/3
    step_header(1, 3, "FFmpeg + COLMAP", icon="🎬")
    run_colmap_processing(video_path, doc_id, colmap_dir)

    # Adım 2/3
    step_header(2, 3, "3D Gaussian Splatting Eğitimi", icon="🧠")
    train_splatfacto_model(doc_id=doc_id, colmap_dir=colmap_dir)

    # Adım 3/3
    step_header(3, 3, "Model Dışa Aktarma (.ply)", icon="💾")
    config_search_path = f"{base_dir}/outputs/colmap_data/splatfacto/*/config.yml"
    try:
        config_path = glob.glob(config_search_path)[0]
    except IndexError:
        raise Exception("Eğitim tamamlandı ama config.yml bulunamadı.")

    subprocess.run([
        "ns-export", "gaussian-splat",
        "--load-config", config_path,
        "--output-dir", export_dir
    ], check=True, cwd=base_dir)

    final_ply_path = os.path.join(export_dir, "splat.ply")
    done_msg(f"3D model üretildi: {os.path.basename(final_ply_path)}")

    return final_ply_path

In [ ]:
def train_splatfacto_model(doc_id, colmap_dir):
    """3D Gaussian Splatting (Splatfacto) modelini eğitir."""
    config_search_path = f"{WORKSPACE_DIR}/{doc_id}/outputs/colmap_data/splatfacto/*/config.yml"
    existing_configs = glob.glob(config_search_path)

    if existing_configs:
        done_msg("Eğitilmiş model zaten mevcut, atlanıyor.")
        return

    try:
        custom_env = os.environ.copy()
        custom_env["TORCH_COMPILE_DISABLE"] = "1"
        custom_env["MAX_JOBS"] = "2"
        abs_colmap_dir = os.path.abspath(colmap_dir)
        splat_result = subprocess.run([
            "ns-train", "splatfacto",
            "--data", abs_colmap_dir,
            "--max-num-iterations", "7000",
            "--vis", "tensorboard"
        ], check=True, capture_output=True, text=True, env=custom_env, cwd=f"{WORKSPACE_DIR}/{doc_id}")
        done_msg("3DGS eğitimi tamamlandı.")
    except subprocess.CalledProcessError as e:
        err_msg(f"3DGS eğitimi başarısız! (exit: {e.returncode})")
        stderr_lines = (e.stderr or "").strip().split('\n')
        for line in stderr_lines[-5:]:
            print(f"  {line}")
        raise Exception(f"ns-train komutu başarısız oldu: {e}")

In [ ]:
import uuid
import urllib.parse
from firebase_admin import firestore, storage

def upload_metrics_and_update_db(local_image_path, splat_model_path, ply_model_path, doc_id, height, diameter, volume, fractal_score, scale_factor, ground_z, qr_code="unknown", additional_metrics=None, color_metrics=None):
    """Firebase Storage'a yükler ve Firestore'u günceller."""
    bucket = storage.bucket()
    db = firestore.client()

    upload_pbar = tqdm(total=4, desc="☁️  Firebase Yükleme", bar_format='{l_bar}{bar:25}{r_bar}')

    # 1. SPLAT model
    cloud_model_path = f"models/{doc_id}_model.splat"
    out_blob = bucket.blob(cloud_model_path)
    out_blob.upload_from_filename(splat_model_path)
    model_token = str(uuid.uuid4())
    out_blob.metadata = {"firebaseStorageDownloadTokens": model_token}
    out_blob.patch()
    encoded_model_path = urllib.parse.quote(cloud_model_path, safe='')
    model_public_url = f"https://firebasestorage.googleapis.com/v0/b/{bucket.name}/o/{encoded_model_path}?alt=media&token={model_token}"
    upload_pbar.update(1)

    # 2. PLY model
    cloud_ply_path = f"models/{doc_id}_model.ply"
    ply_blob = bucket.blob(cloud_ply_path)
    ply_blob.upload_from_filename(ply_model_path)
    ply_token = str(uuid.uuid4())
    ply_blob.metadata = {"firebaseStorageDownloadTokens": ply_token}
    ply_blob.patch()
    encoded_ply_path = urllib.parse.quote(cloud_ply_path, safe='')
    ply_public_url = f"https://firebasestorage.googleapis.com/v0/b/{bucket.name}/o/{encoded_ply_path}?alt=media&token={ply_token}"
    upload_pbar.update(1)

    # 3. Metrik görüntüsü
    destination_blob_name = f"trees/{doc_id}/metrics_view.jpg"
    blob = bucket.blob(destination_blob_name)
    blob.upload_from_filename(local_image_path)
    image_token = str(uuid.uuid4())
    blob.metadata = {"firebaseStorageDownloadTokens": image_token}
    blob.patch()
    encoded_image_path = urllib.parse.quote(destination_blob_name, safe='')
    image_public_url = f"https://firebasestorage.googleapis.com/v0/b/{bucket.name}/o/{encoded_image_path}?alt=media&token={image_token}"
    upload_pbar.update(1)

    # 4. Firestore güncelleme
    measurements = {
        "height_m": round(height, 2),
        "diameter_cm": round(diameter, 1),
        "volume_m3": round(volume, 2),
        "fractal_score": round(fractal_score, 3)
    }
    if additional_metrics:
        measurements.update({
            "crown_diameter_m": additional_metrics.get("crown_diameter_m", 0.0),
            "crown_surface_area_m2": additional_metrics.get("crown_surface_area_m2", 0.0),
            "trunk_lean_deg": additional_metrics.get("trunk_lean_deg", 0.0),
            "crown_asymmetry_index": additional_metrics.get("crown_asymmetry_index", 0.0),
            "crown_density_pts_m3": additional_metrics.get("crown_density_pts_m3", 0.0),
            "crown_base_height_m": additional_metrics.get("crown_base_height_m", 0.0),
        })

    tree_data = {
        "status": "completed",
        "qr_code": qr_code,
        "measurements": measurements,
        "transform": {"scale": scale_factor, "z_offset": float(ground_z)},
        'model_url': model_public_url,
        'ply_url': ply_public_url,
        "metrics_image_url": image_public_url,
        "last_updated": firestore.SERVER_TIMESTAMP
    }
    if color_metrics:
        tree_data["color_analysis"] = {
            "green_fraction": color_metrics.get("green_fraction", 0.0),
            "greenness_index": color_metrics.get("greenness_index", 0.0),
            "stress_ratio": color_metrics.get("stress_ratio", 0.0),
            "color_homogeneity": color_metrics.get("color_homogeneity", 0.0),
            "trunk_leaf_ratio": color_metrics.get("trunk_leaf_ratio", 0.0),
            "health_score": color_metrics.get("health_score", 0),
            "dominant_color_rgb": color_metrics.get("dominant_color_rgb", [0, 0, 0]),
            "analyzed_frames": color_metrics.get("analyzed_frames", 0)
        }

    db.collection("trees").document(doc_id).set(tree_data, merge=True)
    upload_pbar.update(1)
    upload_pbar.close()

    summary_box("Firebase Yükleme", [
        ("Doküman", doc_id),
        ("QR Kod", qr_code),
        ("SPLAT", "✅ Yüklendi"),
        ("PLY", "✅ Yüklendi"),
        ("Metrik Görüntü", "✅ Yüklendi"),
        ("Firestore", "✅ Güncellendi"),
    ], color="#f57c00", icon="☁️")

    return image_public_url, model_public_url, ply_public_url

In [ ]:
def is_video_h264_mp4(video_path):
    """Videoun H264 codec ve yuv420p piksel formatında olup olmadığını kontrol eder."""
    try:
        cmd = [
            "ffprobe",
            "-v", "error",
            "-select_streams", "v:0",
            "-show_entries", "stream=codec_name,pix_fmt",
            "-of", "json",
            video_path
        ]
        result = subprocess.run(cmd, check=True, capture_output=True, text=True)
        info = json.loads(result.stdout)

        if "streams" in info and len(info["streams"]) > 0:
            video_stream = info["streams"][0]
            if video_stream.get("codec_name") == "h264" and video_stream.get("pix_fmt") == "yuv420p":
                return True
        return False
    except (subprocess.CalledProcessError, json.JSONDecodeError) as e:
        print(f"[WARNING] ffprobe failed for {video_path}: {e}")
        return False # ffprobe başarısız olursa, formatın istenen gibi olmadığını varsayalım

In [ ]:
def create_2d_metrics_view(pcd, doc_id, height, diameter, volume):
    """
    3D nokta bulutunu kullanarak yan yana (Kuş Bakışı ve Yandan Görünüm)
    iki planlı bir 2D metrik tablosu (Blueprint) oluşturur.
    """
    # 1. Nokta Bulutunu Yükle ve Noktaları Al
    points = np.asarray(pcd.points)

    # 2. AYARLAR VE TUVAL BOYUTLARI
    ppm = 150
    margin = 1.0

    x_min, x_max = np.min(points[:, 0]), np.max(points[:, 0])
    y_min, y_max = np.min(points[:, 1]), np.max(points[:, 1])
    z_min, z_max = np.min(points[:, 2]), np.max(points[:, 2])

    top_w = int((x_max - x_min + 2*margin) * ppm)
    top_h = int((y_max - y_min + 2*margin) * ppm)
    side_w = int((x_max - x_min + 2*margin) * ppm)
    side_h = int((z_max - z_min + 2*margin) * ppm)

    canvas_h = max(top_h, side_h)
    canvas_w = top_w + side_w

    metrics_img = np.ones((canvas_h, canvas_w, 3), dtype=np.uint8) * 255

    c_tree = (34, 139, 34)
    c_measure = (255, 100, 0)
    c_text = (50, 50, 50)
    c_ground = (40, 70, 130)

    # 3. SOL: KUŞ BAKIŞI (TOP VIEW)
    px_top = ((points[:, 0] - x_min + margin) * ppm).astype(int)
    py_top = ((points[:, 1] - y_min + margin) * ppm).astype(int)
    metrics_img[py_top, px_top] = c_tree

    cx_top = int((0 - x_min + margin) * ppm)
    cy_top = int((0 - y_min + margin) * ppm)
    cv2.drawMarker(metrics_img, (cx_top, cy_top), c_measure, markerType=cv2.MARKER_CROSS, markerSize=20, thickness=2)
    cv2.putText(metrics_img, "KUS BAKISI (XY)", (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.8, c_text, 2)
    cv2.putText(metrics_img, f"Tac Hacmi: {volume:.2f} m3", (20, 80), cv2.FONT_HERSHEY_SIMPLEX, 0.7, c_measure, 2)
    cv2.line(metrics_img, (top_w, 0), (top_w, canvas_h), (200, 200, 200), 2)

    # 4. SAĞ: YANDAN GÖRÜNÜM (SIDE VIEW)
    px_side = ((points[:, 0] - x_min + margin) * ppm + top_w).astype(int)
    py_side = (canvas_h - (points[:, 2] - z_min + margin) * ppm).astype(int)
    metrics_img[py_side, px_side] = c_tree

    ground_y = int(canvas_h - (0 - z_min + margin) * ppm)
    cv2.line(metrics_img, (top_w + 20, ground_y), (canvas_w - 20, ground_y), c_ground, 3)
    cv2.putText(metrics_img, "Zemin (Z=0)", (top_w + 20, ground_y + 25), cv2.FONT_HERSHEY_SIMPLEX, 0.6, c_ground, 2)

    cx_side = int((0 - x_min + margin) * ppm + top_w)

    top_y = int(canvas_h - (height - z_min + margin) * ppm)
    offset_x = 80
    cv2.arrowedLine(metrics_img, (cx_side + offset_x, ground_y), (cx_side + offset_x, top_y), c_measure, 2, tipLength=0.05)
    cv2.arrowedLine(metrics_img, (cx_side + offset_x, top_y), (cx_side + offset_x, ground_y), c_measure, 2, tipLength=0.05)
    cv2.putText(metrics_img, f"Boy: {height:.2f} m", (cx_side + offset_x + 10, int((ground_y + top_y)/2)), cv2.FONT_HERSHEY_SIMPLEX, 0.7, c_measure, 2)

    trunk_y = int(canvas_h - (0.4 - z_min + margin) * ppm)
    radius_px = int((diameter / 200) * ppm)
    cv2.arrowedLine(metrics_img, (cx_side - radius_px - 30, trunk_y), (cx_side - radius_px, trunk_y), c_measure, 2)
    cv2.putText(metrics_img, f"Cap: {diameter:.1f} cm", (cx_side - radius_px - 140, trunk_y + 5), cv2.FONT_HERSHEY_SIMPLEX, 0.6, c_measure, 2)
    cv2.putText(metrics_img, "YANDAN GORUNUM (XZ)", (top_w + 20, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.8, c_text, 2)

    # 5. KAYDET
    os.makedirs(f"{WORKSPACE_DIR}/{doc_id}/outputs", exist_ok=True)
    metrics_view_path = f"{WORKSPACE_DIR}/{doc_id}/{doc_id}_metrics.jpg"
    cv2.imwrite(metrics_view_path, metrics_img)
    done_msg(f"2D Mimari Plan kaydedildi ({canvas_w}×{canvas_h}px)")

    return metrics_view_path

In [ ]:
def process_pending_trees():
    """Firestore'da 'bekliyor' (pending) durumundaki ağaçları bulur ve işler."""
    pending_docs = list(db.collection('trees').where('status', '==', 'pending').stream())

    if not pending_docs:
        warn_msg("Bekleyen ağaç bulunamadı.")
        return

    done_msg(f"{len(pending_docs)} bekleyen ağaç bulundu.")

    for doc in pending_docs:
        tree_data = doc.to_dict()
        doc_id = doc.id
        video_url_in_storage = tree_data.get('video_url')
        TOTAL_STEPS = 10

        display(HTML(f"""<div style='font-family:"SF Mono",monospace;font-size:14px;font-weight:700;
            color:#1565c0;padding:10px 0 4px;border-bottom:2px solid #1565c0;margin:12px 0 6px'>
            🌲 İşleniyor: <code>{doc_id}</code></div>"""))

        db.collection('trees').document(doc_id).update({'status': 'processing'})

        try:
            if not video_url_in_storage:
                raise Exception("Video yolu Firebase'de bulunamadı veya boş!")

            parsed_url = urllib.parse.urlparse(video_url_in_storage)
            path_segments = parsed_url.path.split('/')
            blob_name_encoded = '/'.join(path_segments[path_segments.index('o') + 1:])
            blob_name = urllib.parse.unquote(blob_name_encoded)

            # ── Adım 1/10: Video İndirme ──
            step_header(1, TOTAL_STEPS, "Video İndirme", icon="📥")
            os.makedirs(WORKSPACE_DIR, exist_ok=True)
            local_video_path = os.path.join(WORKSPACE_DIR, f"{doc_id}.mp4")
            blob = bucket.blob(blob_name)
            blob.download_to_filename(local_video_path)
            done_msg("Video indirildi.")

            # ── Adım 2/10: Codec Kontrolü ──
            step_header(2, TOTAL_STEPS, "Video Format Kontrolü", icon="🎬")
            reencoded_video_path = os.path.join(WORKSPACE_DIR, f"{doc_id}_reencoded.mp4")
            if is_video_h264_mp4(local_video_path):
                done_msg("Video zaten H264 MP4 formatında.")
                video_to_analyze = local_video_path
            else:
                try:
                    subprocess.run([
                        "ffmpeg", "-y", "-i", local_video_path,
                        "-vcodec", "libx264", "-pix_fmt", "yuv420p",
                        "-preset", "medium", "-crf", "23",
                        reencoded_video_path
                    ], check=True, capture_output=True, text=True)
                    done_msg("Video yeniden kodlandı.")
                    video_to_analyze = reencoded_video_path
                except (subprocess.CalledProcessError, FileNotFoundError) as e:
                    warn_msg(f"Yeniden kodlama başarısız, orijinal kullanılacak.")
                    video_to_analyze = local_video_path

            # ── Adım 3/10: QR + ArUco Tarama ──
            step_header(3, TOTAL_STEPS, "QR Kod + ArUco Tarama", icon="🔍")
            qr_data, marker_pixel_width, detected_frame_count, aruco_observations = analyze_video_for_hybrid_plate(video_to_analyze)

            if qr_data is None:
                qr_data = "unknown"
                warn_msg("QR Kod bulunamadı → 'unknown'")
            db.collection('trees').document(doc_id).update({'qr_code': qr_data})

            if marker_pixel_width is None:
                err_msg("ArUco Marker bulunamadı!")

            # ── Adım 4/10: Renk Analizi ──
            step_header(4, TOTAL_STEPS, "Video Renk Analizi", icon="🎨")
            color_metrics = analyze_video_color(video_to_analyze, n_frames=20)

            # ── Adım 5/10: 3D Rekonstrüksiyon ──
            step_header(5, TOTAL_STEPS, "3D Model Üretimi (COLMAP + 3DGS)", icon="🧊")
            raw_model_path = run_3d_reconstruction_pipeline(video_to_analyze, doc_id)

            # ── Adım 6/10: ArUco Triangülasyon ──
            step_header(6, TOTAL_STEPS, "ArUco 3D Konum Triangülasyonu", icon="📐")
            transforms_path = f"{WORKSPACE_DIR}/{doc_id}/colmap_data/transforms.json"
            aruco_3d_pos = find_aruco_3d_position(transforms_path, aruco_observations)

            # ── Adım 7/10: Ölçek Çarpanı ──
            step_header(7, TOTAL_STEPS, "Ölçek Çarpanı Hesaplama", icon="📏")
            scale_factor = calculate_scale_factor(transforms_path, aruco_observations, marker_pixel_width)

            # ── Adım 8/10: Ölçüm + Gürültü Temizleme ──
            step_header(8, TOTAL_STEPS, "Ağaç Ölçümleri + Nokta Bulutu Temizleme", icon="📊")
            additional_metrics = None
            if marker_pixel_width is not None:
                height, volume, trunk_diameter_cm, fractal_score, ground_z, pcd, additional_metrics = measure_tree_metrics(
                    raw_model_path, scale_factor, aruco_3d_pos=aruco_3d_pos
                )
                local_image_path = create_2d_metrics_view(pcd, doc_id, height, trunk_diameter_cm, volume)

            # ── Adım 9/10: SPLAT Dönüşümü ──
            step_header(9, TOTAL_STEPS, "PLY → SPLAT Dönüşümü", icon="💎")
            splat_model_path = raw_model_path.replace(".ply", ".splat")
            convert_ply_to_splat(
                raw_model_path, splat_model_path,
                aruco_3d_pos=aruco_3d_pos, scale_factor=scale_factor
            )

            # ── Adım 10/10: Firebase Yükleme ──
            step_header(10, TOTAL_STEPS, "Firebase Yükleme", icon="☁️")
            upload_metrics_and_update_db(
                local_image_path=local_image_path,
                splat_model_path=splat_model_path,
                ply_model_path=raw_model_path,
                doc_id=doc_id, height=height,
                diameter=trunk_diameter_cm, volume=volume,
                fractal_score=fractal_score, scale_factor=scale_factor,
                ground_z=ground_z, qr_code=qr_data,
                additional_metrics=additional_metrics,
                color_metrics=color_metrics
            )

            display(HTML(f"""<div style='font-family:"SF Mono",monospace;font-size:13px;font-weight:700;
                color:#2d7d46;padding:8px 12px;margin:6px 0;background:#e8f5e9;border-radius:6px;
                border-left:4px solid #2d7d46'>🎉 {doc_id} başarıyla tamamlandı!</div>"""))

        except Exception as e:
            err_msg(f"{doc_id} işlenirken hata: {str(e)}")
            db.collection('trees').document(doc_id).update({'status': 'pending', 'error': str(e)})

In [ ]:
# --- CERRAHİ YAMA: PyTorch 3.12 Dynamo Hatasını Fiziksel Olarak Kapat ---
patch_file_path = "/usr/local/lib/python3.12/dist-packages/nerfstudio/utils/misc.py"
if os.path.exists(patch_file_path):
    with open(patch_file_path, "r") as f:
        content = f.read()

    # Hata veren kodu, boş dönen bir lambda fonksiyonuyla değiştir
    old_code = "return torch.compile(*args, **kwargs)"
    new_code = "return lambda x: x"

    if old_code in content:
        content = content.replace(old_code, new_code)
        with open(patch_file_path, "w") as f:
            f.write(content)
        print("[SİSTEM] Nerfstudio Dynamo hatası başarıyla yamalandı!")
# -----------------------------------------------------------------------

# Nerfstudio'nun model yükleme dosyasının yolunu buluyoruz
file_path = '/usr/local/lib/python3.12/dist-packages/nerfstudio/utils/eval_utils.py'

# Dosyayı okuyoruz
with open(file_path, 'r') as file:
    content = file.read()

# PyTorch 2.6'nın güvenlik duvarını aşacak parametreyi koda zorla enjekte ediyoruz
eski_kod = 'torch.load(load_path, map_location="cpu")'
yeni_kod = 'torch.load(load_path, map_location="cpu", weights_only=False)'
content = content.replace(eski_kod, yeni_kod)

# Yamalı dosyayı geri kaydediyoruz
with open(file_path, 'w') as file:
    file.write(content)

print("Nerfstudio, PyTorch 2.6 güvenlik duvarına karşı başarıyla yamalandı!")

In [ ]:
import os
import subprocess

gpu_colmap_path = "/kaggle/working/gpu_colmap"
wrapper_dir = "/kaggle/working/colmap_wrapper"
os.makedirs(wrapper_dir, exist_ok=True)

if not os.path.exists(gpu_colmap_path) or not os.path.exists(f"{gpu_colmap_path}/bin/colmap"):
    print("1. Standalone paket yöneticisi (Micromamba) indiriliyor...")
    subprocess.run("curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xvj bin/micromamba", shell=True)
    
    print("2. COLMAP indiriliyor (Kısıtlamasız Kök CUDA Versiyonu İsteniyor)...")
    # KRİTİK DEĞİŞİKLİK: cudatoolkit=11.8 KISITLAMASINI SİLDİK! Artık kendi 12.6+ versiyonunu özgürce çekecek.
    subprocess.run(f"./bin/micromamba create -p {gpu_colmap_path} -c conda-forge \"colmap=*=*cuda*\" -y", shell=True)
    
print("\n--- COLMAP GPU Kurulum Bitti. Tercüman (Wrapper) ayarlanıyor... ---")

# TERCÜMAN WRAPPER: Eski Nerfstudio komutlarını Yeni GPU COLMAP diline çevirir
wrapper_script = """#!/bin/bash
ARGS=("$@")
for i in "${!ARGS[@]}"; do
    if [[ "${ARGS[$i]}" == "--SiftExtraction.use_gpu" ]]; then
        ARGS[$i]="--FeatureExtraction.use_gpu"
    elif [[ "${ARGS[$i]}" == "--SiftMatching.use_gpu" ]]; then
        ARGS[$i]="--FeatureMatching.use_gpu"
    fi
done
# Çevrilmiş parametrelerle GERÇEK GPU'LU COLMAP'i çalıştır!
/kaggle/working/gpu_colmap/bin/colmap "${ARGS[@]}"
"""

wrapper_path = f"{wrapper_dir}/colmap"
with open(wrapper_path, "w") as f:
    f.write(wrapper_script)
os.chmod(wrapper_path, 0o755)

# PATH'in en başına sadece tercümanın olduğu klasörü koyuyoruz
current_path = os.environ.get('PATH', '')
clean_path = current_path.replace(f"{gpu_colmap_path}/bin:", "")
os.environ["PATH"] = f"{wrapper_dir}:{clean_path}"

print("--- Tüm sistem başarıyla Tercümana yönlendirildi ve GERÇEK CUDA kütüphaneleri aktif edildi! ---")

In [52]:
# --- İŞÇİYİ BAŞLAT ---
if __name__ == "__main__":
    # Colab defterinde bu hücreyi çalıştırdığınızda bekleyen işleri tarar
    process_pending_trees()

Firebase kontrol ediliyor...
Yeni iş bulundu: mhehdjn1fBIAbRLylWdR. İşlem başlatılıyor...
Depolama yolu: https://firebasestorage.googleapis.com/v0/b/studio-166104040-52130.firebasestorage.app/o/videos%2Fgemini1.mp4?alt=media&token=307a2dc7-3277-41aa-944f-e090ba792653


/usr/local/lib/python3.12/dist-packages/google/cloud/firestore_v1/base_collection.py:316: UserWarning: Detected filter using positional arguments. Prefer using the 'filter' keyword argument instead.
  return query.where(field_path, op_string, value)


Video indirildi.
[DEBUG] Video /kaggle/working/pistachio_workspace/mhehdjn1fBIAbRLylWdR.mp4 zaten H264 MP4 (yuv420p) formatında. Yeniden kodlama atlanıyor.
[DEBUG] Analiz edilecek video yolu: /kaggle/working/pistachio_workspace/mhehdjn1fBIAbRLylWdR.mp4
[DEBUG] Hibrit plaka analizi başlıyor: /kaggle/working/pistachio_workspace/mhehdjn1fBIAbRLylWdR.mp4
[DEBUG] Frame 21'te ArUco Marker Bulundu!
[DEBUG] ArUco Piksel Genişliği: 19.31
HATA: QR Kod veya ArUco Marker videonun başında net olarak bulunamadı!
[mhehdjn1fBIAbRLylWdR] GERÇEK 3D ÜRETİM BAŞLIYOR...
[mhehdjn1fBIAbRLylWdR] Adım 1/3: FFmpeg ve COLMAP çalışıyor (Bu adım videonun uzunluğuna göre 5-15 dk sürer)...
[DEBUG] COLMAP veritabanı /kaggle/working/pistachio_workspace/mhehdjn1fBIAbRLylWdR/colmap_data/colmap/database.db zaten mevcut. Adım 1 atlanıyor.
[mhehdjn1fBIAbRLylWdR] Adım 2/3: 3DGS modeli eğitiliyor (GPU tam kapasite devrede, 15-25 dk sürer)....
[DEBUG] Eğitilmiş model /kaggle/working/pistachio_workspace/mhehdjn1fBIAbRLylWdR/ou

2026-03-03 20:58:51.083803: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772571531.106411    3293 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772571531.113476    3293 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772571531.132108    3293 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772571531.132137    3293 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772571531.132139    3293 computation_placer.cc:177] computation placer alr

Loading latest checkpoint from load_dir
✅ Done loading checkpoint from outputs/colmap_data/splatfacto/2026-03-03_191814/nerfstudio_models/step-000006999.ckpt
0 Gaussians have NaN/Inf and 7 have low opacity, only export 349824/349831
[mhehdjn1fBIAbRLylWdR] GERÇEK 3D ÜRETİM TAMAMLANDI. Çıktı: /kaggle/working/pistachio_workspace/mhehdjn1fBIAbRLylWdR/export/splat.ply
[DEBUG] Ölçek Çarpanı (Scale Factor) hesaplanıyor...
[DEBUG] Kameranın ağaca GERÇEK mesafesi: 3.02 metre
UYARI: ArUco bulunan kare transforms.json içinde bulunamadı! Varsayılan çarpan 1.0 kullanılıyor.
[DEBUG] 3D Model analiz ediliyor: /kaggle/working/pistachio_workspace/mhehdjn1fBIAbRLylWdR/export/splat.ply
[DEBUG] Uygulanan Ölçek Çarpanı: 1.0000
[SONUÇ] Gövde Çapı (30-50cm arası): 2096.0 cm
[DEBUG] Taç apısının Fraktal Skoru hesaplanıyor...
[SONUÇ] Ağacın Fraktal Skoru (D_f): 1.881
[SONUÇ] Ağaç Boyu: 16.62 Metre
[SONUÇ] Taç Hacmi: 7232.65 Metreküp
[SONUÇ] Ağaç Gövde Çapı: 2096.04 Santimetre
[SONUÇ] Ağaç Fraktal Boyutu: 1.88 

In [ ]:
#!rm -r /kaggle/working/pistachio_workspace/mhehdjn1fBIAbRLylWdR
#!rm -r /kaggle/working/gpu_colmap

In [37]:
#!rm -r /kaggle/working/pistachio_workspace/mhehdjn1fBIAbRLylWdR/export